# Stage 3: CIEI-MERC — IEMOCAP
**Counterfactual Attention Drop (CAD) implicit edge inference**  
Novel visual stream: temporal 3-segment (begin/mid/end), OpenFace AU always present alongside CLIP or SigLIP2.  
Backbone: alternating inter/intra-modal SGConv (pure PyTorch).  
Loss: CB-Focal + EACL anchor-contrastive.

In [ ]:
import os, sys, random, json
from collections import defaultdict
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, classification_report

def seed_everything(seed=42):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
seed_everything(42)

# ── Paths ─────────────────────────────────────────────────────────────────────
BASE      = '/mnt/Work/ML/Thesis/BMVC/Hopeful/Dataset/Processed/IEMOCAP'
FEAT_DIR  = os.path.join(BASE, 'features')
LABELS    = os.path.join(BASE, 'labels.csv')
CKPT_DIR  = '/mnt/Work/ML/Thesis/BMVC/Hopeful/checkpoints/iemocap_ciei'
os.makedirs(CKPT_DIR, exist_ok=True)

# ── Visual config (swap here for ablations) ───────────────────────────────────
vis_backbone = 'siglip2'   # 'clip' | 'siglip2'
vis_pool_mode = 'attn'     # 'attn' | 'concat'

# ── Model ─────────────────────────────────────────────────────────────────────
hidden_dim   = 256
n_layers     = 4       # alternating SGConv layers (intra→inter→intra→inter)
drop         = 0.3

# ── Graph ─────────────────────────────────────────────────────────────────────
window_k     = 10     # temporal adjacency half-window
cad_k        = 3      # top-k implicit edges per destination node
cad_tau      = 0.5    # Gumbel-Softmax temperature
cad_attn_dim = 128    # CAD attention key/query dim

# ── Ablation flags ────────────────────────────────────────────────────────────
use_cad   = True
use_eacl  = True
use_focal = True

# ── Loss weights ─────────────────────────────────────────────────────────────
eacl_weight  = 0.5
focal_weight = 1.0
focal_gamma  = 2.0

# ── Training ──────────────────────────────────────────────────────────────────
lr           = 1e-4
weight_decay = 1e-4
epochs       = 60
patience     = 10

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"device={device}  vis_backbone={vis_backbone}  vis_pool={vis_pool_mode}")
print(f"use_cad={use_cad}  use_eacl={use_eacl}  use_focal={use_focal}")

## §2 Data Loading
IEMOCAP 6-class: `neu hap exc sad ang fru`. Sessions 1–4 train, Session 5 test.

In [ ]:
VALID_EMOTIONS = {'neu', 'hap', 'exc', 'sad', 'ang', 'fru'}
EMO2IDX = {'neu': 0, 'hap': 1, 'exc': 2, 'sad': 3, 'ang': 4, 'fru': 5}
IDX2EMO = {v: k for k, v in EMO2IDX.items()}
N_CLASSES = len(EMO2IDX)

print("Loading features (this may take ~30s)...")
text_feats  = torch.load(os.path.join(FEAT_DIR, 'text_roberta_large.pt'),            weights_only=False)
audio_feats = torch.load(os.path.join(FEAT_DIR, 'audio_microsoft_wavlm_large.pt'),   weights_only=False)

if vis_backbone == 'clip':
    vis_feats   = torch.load(os.path.join(FEAT_DIR, 'video_clip_temporal.pt'),        weights_only=False)
    VIS_SEM_DIM = 768
else:
    vis_feats   = torch.load(os.path.join(FEAT_DIR, 'video_siglip2_temporal.pt'),     weights_only=False)
    VIS_SEM_DIM = 1152

au_feats = torch.load(os.path.join(FEAT_DIR, 'video_openface_au.pt'), weights_only=False)

# Sanity check shapes
_k = list(text_feats.keys())[0]
assert np.array(text_feats[_k]).shape  == (1024,),          f"text shape wrong: {np.array(text_feats[_k]).shape}"
assert np.array(audio_feats[_k]).shape == (1024,),          f"audio shape wrong"
assert np.array(vis_feats[_k]).shape   == (3, VIS_SEM_DIM), f"vis shape wrong: {np.array(vis_feats[_k]).shape}"
assert np.array(au_feats[_k]).shape    == (3, 8),           f"AU shape wrong"
print(f"Features OK — text/audio (1024,) | vis (3,{VIS_SEM_DIM}) | au (3,8)")

In [ ]:
def build_dialogues(split='train'):
    """
    Group utterances by dialogue, filter to 6 classes.
    Returns list of dicts: {text, audio, vis_sem, au, speakers, labels, dia_id, N}
    """
    df = pd.read_csv(LABELS)
    df = df[df['emotion'].isin(VALID_EMOTIONS)].copy()

    if split == 'train':
        df = df[df['session'] != 'Session5']
    else:
        df = df[df['session'] == 'Session5']

    dialogues, skipped = [], 0
    for dia_id, grp in df.groupby('dialog'):
        grp = grp.sort_values('utt_id').reset_index(drop=True)
        uids = grp['utt_id'].tolist()

        # Drop dialogue if any utterance missing in ANY feature dict
        if any(uid not in text_feats or uid not in vis_feats or uid not in au_feats
               for uid in uids):
            skipped += 1
            continue

        dialogues.append({
            'text':     torch.FloatTensor(np.stack([np.array(text_feats[u])  for u in uids])),  # (N,1024)
            'audio':    torch.FloatTensor(np.stack([np.array(audio_feats[u]) for u in uids])),  # (N,1024)
            'vis_sem':  torch.FloatTensor(np.stack([np.array(vis_feats[u])   for u in uids])),  # (N,3,VIS_SEM_DIM)
            'au':       torch.FloatTensor(np.stack([np.array(au_feats[u])    for u in uids])),  # (N,3,8)
            'speakers': grp['speaker'].tolist(),
            'labels':   torch.LongTensor([EMO2IDX[e] for e in grp['emotion']]),
            'dia_id':   dia_id,
            'N':        len(uids),
        })

    total_utts = sum(d['N'] for d in dialogues)
    print(f"[{split}] {len(dialogues)} dialogues | {total_utts} utterances | {skipped} skipped")
    return dialogues

train_dias = build_dialogues('train')
test_dias  = build_dialogues('test')

# Class counts for focal loss
from collections import Counter
train_labels_flat = [l.item() for d in train_dias for l in d['labels']]
class_counts = [Counter(train_labels_flat).get(i, 1) for i in range(N_CLASSES)]
print("Class counts:", {IDX2EMO[i]: class_counts[i] for i in range(N_CLASSES)})

## §3 Graph Utilities
**Explicit edges** (proven, no novelty): temporal adjacency ±k, same-speaker, cross-modal triadic.  
**CAD implicit edges** (novel): influence score = L2 difference of attention output with vs without utterance i in context.

In [ ]:
def build_explicit_edges(N, speakers, k=10):
    """
    Build explicit edges for one dialogue of length N.
    Node layout: text=[0..N-1]  audio=[N..2N-1]  visual=[2N..3N-1]

    Edge types:
      1. Temporal adjacency (intra-modal): |i-j| <= k, both directions
      2. Same-speaker (intra-modal): same speaker, both directions
      3. Same-utterance cross-modal (t_i <-> a_i <-> v_i), bidirectional
    """
    srcs, dsts = [], []

    for off in [0, N, 2*N]:
        # 1. Temporal adjacency
        for i in range(N):
            for j in range(max(0, i - k), min(N, i + k + 1)):
                if i != j:
                    srcs.append(off + i); dsts.append(off + j)
        # 2. Same-speaker
        for i in range(N):
            for j in range(N):
                if i != j and speakers[i] == speakers[j]:
                    srcs.append(off + i); dsts.append(off + j)

    # 3. Cross-modal triadic (bidirectional for all pairs)
    for i in range(N):
        ti, ai, vi = i, N + i, 2 * N + i
        for s, d in [(ti,ai),(ai,ti),(ti,vi),(vi,ti),(ai,vi),(vi,ai)]:
            srcs.append(s); dsts.append(d)

    return torch.LongTensor(srcs), torch.LongTensor(dsts)


def _get_backward_nbrs(srcs, dsts, offset, N):
    """Extract backward (past) intra-modal neighbours per node for CAD."""
    nbrs = [[] for _ in range(N)]
    for s, d in zip(srcs.tolist(), dsts.tolist()):
        ls, ld = s - offset, d - offset
        if 0 <= ls < N and 0 <= ld < N and ls < ld:
            nbrs[ld].append(ls)   # ls is a past neighbour of ld
    return nbrs

In [ ]:
class CADModule(nn.Module):
    """
    Counterfactual Attention Drop — pseudo-causal implicit edge inference.

    For each forward-looking pair (i -> j, j > i) within one modality:
      ctx_j         = explicit backward neighbours of j
      H_j_full      = f_attn(H_j | ctx_j)
      H_j_masked    = f_attn(H_j | ctx_j \ {H_i})
      Inf(i -> j)   = ||H_j_full - H_j_masked||_2   (stop-grad)

    Top-k edges per destination node selected via Gumbel-Softmax.
    """

    def __init__(self, hidden_dim, attn_dim=128, top_k=3, tau=0.5):
        super().__init__()
        self.top_k = top_k
        self.tau   = tau
        self.q  = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.k_ = nn.Linear(hidden_dim, attn_dim, bias=False)
        self.v  = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.scale = attn_dim ** -0.5

    def _attend(self, query, keys, values):
        """query:(D,) keys:(M,D) values:(M,D) -> (D,)"""
        q = self.q(query).unsqueeze(0)          # (1, A)
        k = self.k_(keys)                        # (M, A)
        attn = F.softmax(q @ k.T * self.scale, dim=-1)  # (1, M)
        return (attn @ self.v(values)).squeeze(0)         # (D,)

    def forward(self, H, N, nbrs_per_node):
        """
        H             : (N, D) detached node features for ONE modality
        N             : dialogue length
        nbrs_per_node : List[List[int]] — backward intra-modal neighbours
        Returns       : (srcs, dsts) LongTensors of implicit edges (offsets NOT added here)
        """
        by_j = defaultdict(list)   # j -> [(i, score)]

        with torch.no_grad():
            for j in range(1, N):
                ctx_idx = nbrs_per_node[j]
                if len(ctx_idx) < 2:
                    continue
                ctx_h    = H[ctx_idx]                          # (M, D)
                H_j_full = self._attend(H[j], ctx_h, ctx_h)

                for i in ctx_idx:
                    remaining = [nb for nb in ctx_idx if nb != i]
                    if not remaining:
                        score = 0.0
                    else:
                        ctx_m  = H[remaining]
                        H_j_m  = self._attend(H[j], ctx_m, ctx_m)
                        score  = (H_j_full - H_j_m).norm().item()
                    by_j[j].append((i, score))

        if not by_j:
            return torch.LongTensor([]), torch.LongTensor([])

        sel_srcs, sel_dsts = [], []
        for j, group in by_j.items():
            k = min(self.top_k, len(group))
            scores = torch.tensor([s for _, s in group], dtype=torch.float)
            # Gumbel-Softmax straight-through top-k
            u = torch.rand_like(scores).clamp(1e-10, 1 - 1e-10)
            perturbed = (scores + -torch.log(-torch.log(u))) / self.tau
            _, topk_idx = perturbed.topk(k)
            for idx in topk_idx.tolist():
                sel_srcs.append(group[idx][0])
                sel_dsts.append(j)

        return torch.LongTensor(sel_srcs), torch.LongTensor(sel_dsts)

In [ ]:
def build_adj(srcs, dsts, n_nodes, dev):
    """Symmetric-normalised adjacency with self-loops: D^{-1/2}(A+I)D^{-1/2}"""
    sl = torch.arange(n_nodes, device=dev)
    all_s = torch.cat([srcs.to(dev), dsts.to(dev), sl])
    all_d = torch.cat([dsts.to(dev), srcs.to(dev), sl])
    idx  = torch.stack([all_s, all_d])
    vals = torch.ones(idx.size(1), device=dev)
    A    = torch.sparse_coo_tensor(idx, vals, (n_nodes, n_nodes)).coalesce().to_dense()
    rs   = A.sum(1).clamp(min=1e-12)
    d    = torch.diag(rs.pow(-0.5))
    return d @ A @ d


def split_intra_inter(A, N):
    """Split (3N×3N) adj into intra-modal (block-diagonal) and inter-modal."""
    A_intra = A.clone()
    # Zero all cross-modal blocks
    for (r0, r1), (c0, c1) in [((0,N),(N,2*N)), ((0,N),(2*N,3*N)), ((N,2*N),(2*N,3*N))]:
        A_intra[r0:r1, c0:c1] = 0
        A_intra[c0:c1, r0:r1] = 0
    return A_intra, A - A_intra

## §4 Model
`TemporalVisPool` → per-modality `proj` MLPs → CAD → 4-layer alternating SGConv → late fusion → head

In [ ]:
class TemporalVisPool(nn.Module):
    """
    Concatenate AU (3,8) to semantic visual (3,VIS_SEM_DIM) → (3, seg_dim).
    Then either:
      concat: flatten 3 segments -> (3*seg_dim,)
      attn  : AU-driven attention over 3 segments -> (seg_dim,)
    """
    def __init__(self, sem_dim, au_dim=8, mode='attn'):
        super().__init__()
        self.mode    = mode
        self.seg_dim = sem_dim + au_dim
        if mode == 'attn':
            self.gate = nn.Linear(self.seg_dim, 1)

    @property
    def out_dim(self):
        return self.seg_dim * 3 if self.mode == 'concat' else self.seg_dim

    def forward(self, vis_sem, au):
        """vis_sem:(N,3,sem_dim)  au:(N,3,8) -> (N, out_dim)"""
        x = torch.cat([vis_sem, au], dim=-1)   # (N, 3, seg_dim)
        if self.mode == 'concat':
            return x.reshape(x.size(0), -1)
        alpha = torch.softmax(self.gate(x), dim=1)  # (N, 3, 1); AU drives which segment matters
        return (alpha * x).sum(dim=1)               # (N, seg_dim)


class SGConvLayer(nn.Module):
    """D^{-1/2}(A+I)D^{-1/2} H W + H  (sym-norm GCN with residual)"""
    def __init__(self, dim, dropout=0.3):
        super().__init__()
        self.W    = nn.Linear(dim, dim, bias=False)
        self.bn   = nn.BatchNorm1d(dim)
        self.act  = nn.LeakyReLU()
        self.drop = nn.Dropout(dropout)

    def forward(self, H, A):
        return self.drop(self.act(self.bn(A @ self.W(H)))) + H

In [ ]:
class CIEI_MERC(nn.Module):
    def __init__(self, vis_sem_dim):
        super().__init__()
        D = hidden_dim

        # Visual temporal pooling
        self.vis_pool = TemporalVisPool(vis_sem_dim, au_dim=8, mode=vis_pool_mode)
        vis_in = self.vis_pool.out_dim

        # Per-modality projection MLPs -> hidden_dim
        self.proj_t = nn.Sequential(nn.Linear(1024,   D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop))
        self.proj_a = nn.Sequential(nn.Linear(1024,   D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop))
        self.proj_v = nn.Sequential(nn.Linear(vis_in, D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop))

        # CAD modules — one per modality
        if use_cad:
            self.cad_t = CADModule(D, cad_attn_dim, cad_k, cad_tau)
            self.cad_a = CADModule(D, cad_attn_dim, cad_k, cad_tau)
            self.cad_v = CADModule(D, cad_attn_dim, cad_k, cad_tau)

        # Alternating SGConv: intra -> inter -> intra -> inter
        self.gcn = nn.ModuleList([SGConvLayer(D, drop) for _ in range(n_layers)])

        # Late fusion: [t || a || v] -> D
        self.fusion = nn.Sequential(
            nn.Linear(3 * D, D), nn.LayerNorm(D), nn.LeakyReLU(), nn.Dropout(drop)
        )
        self.head = nn.Linear(D, N_CLASSES)

    # ------------------------------------------------------------------
    def forward_dialogue(self, dia, return_embeds=False):
        """Process ONE dialogue. Returns logits (N,C) and optionally fused embeds (N,D)."""
        N  = dia['N']
        dev = next(self.parameters()).device

        # 1. Visual temporal pooling + projection
        h_v_raw = self.vis_pool(dia['vis_sem'].to(dev), dia['au'].to(dev))
        h_t = self.proj_t(dia['text'].to(dev))    # (N, D)
        h_a = self.proj_a(dia['audio'].to(dev))   # (N, D)
        h_v = self.proj_v(h_v_raw)                # (N, D)

        # 2. Explicit edges
        e_s, e_d = build_explicit_edges(N, dia['speakers'], window_k)

        # 3. CAD: add implicit edges per modality
        all_s, all_d = e_s.clone(), e_d.clone()
        if use_cad:
            for h_mod, cad_mod, off in [
                (h_t, self.cad_t, 0),
                (h_a, self.cad_a, N),
                (h_v, self.cad_v, 2 * N),
            ]:
                nbrs = _get_backward_nbrs(e_s, e_d, off, N)
                impl_s, impl_d = cad_mod(h_mod.detach(), N, nbrs)
                if impl_s.numel() > 0:
                    all_s = torch.cat([all_s, impl_s + off])
                    all_d = torch.cat([all_d, impl_d + off])

        # 4. Full adjacency + intra/inter split
        A          = build_adj(all_s, all_d, 3 * N, dev)
        A_intra, A_inter = split_intra_inter(A, N)

        # 5. Alternating GCN
        H = torch.cat([h_t, h_a, h_v], dim=0)   # (3N, D)
        for l, layer in enumerate(self.gcn):
            H = layer(H, A_intra if l % 2 == 0 else A_inter)

        # 6. Late fusion + head
        fused  = self.fusion(torch.cat([H[:N], H[N:2*N], H[2*N:]], dim=-1))  # (N, D)
        logits = self.head(fused)                                               # (N, C)

        return (logits, fused) if return_embeds else logits

    def forward(self, dialogues, return_embeds=False):
        return [self.forward_dialogue(d, return_embeds) for d in dialogues]

# Quick shape test
_m = CIEI_MERC(VIS_SEM_DIM).to(device)
_d = train_dias[0]
_lg, _em = _m.forward_dialogue(_d, return_embeds=True)
assert _lg.shape == (train_dias[0]['N'], N_CLASSES)
assert _em.shape == (train_dias[0]['N'], hidden_dim)
print(f"Model OK — logits {_lg.shape}, embeds {_em.shape}")
print(f"Params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}")
del _m, _lg, _em

## §5 Losses
`CBFocalLoss` for class imbalance. `EACLoss` anchor-contrastive for happy↔excited, anger↔frustrated confusions.

In [ ]:
class CBFocalLoss(nn.Module):
    """Class-Balanced Focal Loss (Cui et al., CVPR 2019)."""
    def __init__(self, class_counts, gamma=2.0, beta=0.9999):
        super().__init__()
        self.gamma = gamma
        n   = torch.tensor(class_counts, dtype=torch.float)
        eff = (1 - beta ** n) / (1 - beta)
        w   = 1.0 / eff
        w   = w / w.sum() * len(class_counts)
        self.register_buffer('w', w)

    def forward(self, logits, targets):
        ce  = F.cross_entropy(logits, targets, weight=self.w, reduction='none')
        pt  = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()


class EACLoss(nn.Module):
    """
    Emotion-Anchor Contrastive Learning (Yu et al., EACL 2023 style).
    Learnable per-class anchor vectors on a hypersphere.
    Pull each sample toward its true-emotion anchor; push from others.
    """
    def __init__(self, n_classes, embed_dim, temperature=0.07):
        super().__init__()
        self.anchors = nn.Parameter(torch.empty(n_classes, embed_dim))
        nn.init.orthogonal_(self.anchors)
        self.T = temperature

    def forward(self, embeds, labels):
        """embeds:(N,D)  labels:(N,) -> scalar loss"""
        a = F.normalize(self.anchors, dim=-1)
        e = F.normalize(embeds,       dim=-1)
        sim = (e @ a.T) / self.T          # (N, C)
        return F.cross_entropy(sim, labels)

## §6 Training

In [ ]:
model      = CIEI_MERC(VIS_SEM_DIM).to(device)
focal_loss = CBFocalLoss(class_counts, gamma=focal_gamma).to(device)
eacl_loss  = EACLoss(N_CLASSES, embed_dim=hidden_dim).to(device)

optimizer  = torch.optim.AdamW(
    list(model.parameters()) + list(eacl_loss.parameters()),
    lr=lr, weight_decay=weight_decay
)
scheduler  = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

print(f"Trainable params: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")
print(f"EACL anchor params: {eacl_loss.anchors.numel()}")

In [ ]:
def run_epoch(dias, train=True):
    model.train(train)
    total_loss, total_n = 0.0, 0
    all_pred, all_true  = [], []

    if train:
        random.shuffle(dias)

    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for dia in dias:
            labels = dia['labels'].to(device)

            if train:
                optimizer.zero_grad()

            logits, embeds = model.forward_dialogue(dia, return_embeds=True)

            # Loss
            loss = torch.tensor(0.0, device=device)
            if use_focal:
                loss = loss + focal_weight * focal_loss(logits, labels)
            else:
                loss = loss + F.cross_entropy(logits, labels)
            if use_eacl:
                loss = loss + eacl_weight * eacl_loss(embeds, labels)

            if train:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(
                    list(model.parameters()) + list(eacl_loss.parameters()), 1.0)
                optimizer.step()

            total_loss += loss.item() * dia['N']
            total_n    += dia['N']
            all_pred.extend(logits.argmax(-1).cpu().tolist())
            all_true.extend(labels.cpu().tolist())

    wf1  = f1_score(all_true, all_pred, average='weighted') * 100
    mf1  = f1_score(all_true, all_pred, average='macro')    * 100
    return total_loss / total_n, wf1, mf1

In [ ]:
import time

best_val_wf1  = 0.0
patience_ctr  = 0
history       = []

print(f"{'Ep':>4} {'Tr-loss':>9} {'Tr-WF1':>8} {'Val-WF1':>9} {'Val-MF1':>9}  time")
print("-" * 60)

for ep in range(1, epochs + 1):
    t0 = time.time()

    tr_loss, tr_wf1, _  = run_epoch(train_dias, train=True)
    val_loss, val_wf1, val_mf1 = run_epoch(test_dias,  train=False)

    scheduler.step()

    elapsed = time.time() - t0
    history.append({'ep': ep, 'tr_loss': tr_loss, 'tr_wf1': tr_wf1,
                    'val_wf1': val_wf1, 'val_mf1': val_mf1})
    print(f"{ep:4d}  {tr_loss:9.4f}  {tr_wf1:7.2f}%  {val_wf1:8.2f}%  {val_mf1:8.2f}%  {elapsed:.0f}s")

    if val_wf1 > best_val_wf1:
        best_val_wf1 = val_wf1
        patience_ctr = 0
        ckpt = {
            'ep': ep, 'model': model.state_dict(),
            'eacl': eacl_loss.state_dict(), 'val_wf1': val_wf1, 'val_mf1': val_mf1,
            'cfg': {
                'vis_backbone': vis_backbone, 'vis_pool_mode': vis_pool_mode,
                'use_cad': use_cad, 'use_eacl': use_eacl, 'use_focal': use_focal,
                'hidden_dim': hidden_dim, 'n_layers': n_layers,
            }
        }
        torch.save(ckpt, os.path.join(CKPT_DIR, 'best.pt'))
        print(f"    ↑ new best  WF1={val_wf1:.2f}%  MF1={val_mf1:.2f}%  (saved)")
    else:
        patience_ctr += 1
        if patience_ctr >= patience:
            print(f"Early stop at epoch {ep} (patience={patience})")
            break

print(f"\nBest val WF1: {best_val_wf1:.2f}%")

## §7 Evaluation

In [ ]:
# Load best checkpoint
ckpt = torch.load(os.path.join(CKPT_DIR, 'best.pt'), weights_only=False)
model.load_state_dict(ckpt['model'])
eacl_loss.load_state_dict(ckpt['eacl'])
print(f"Loaded best model from epoch {ckpt['ep']}  val_WF1={ckpt['val_wf1']:.2f}%")

# Full evaluation
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for dia in test_dias:
        logits = model.forward_dialogue(dia)
        all_pred.extend(logits.argmax(-1).cpu().tolist())
        all_true.extend(dia['labels'].tolist())

wf1 = f1_score(all_true, all_pred, average='weighted') * 100
mf1 = f1_score(all_true, all_pred, average='macro')    * 100
print(f"\nTest  WF1={wf1:.2f}%  MF1={mf1:.2f}%")
print("\nPer-class report:")
print(classification_report(all_true, all_pred, target_names=[IDX2EMO[i] for i in range(N_CLASSES)]))

## §8 Ablations
Re-run with single flag changes. Each result is a fresh model trained from scratch.  
Key question: **how much does CAD add over explicit edges only?**

In [ ]:
# ── Ablation runner ───────────────────────────────────────────────────────────
def run_ablation(label, **overrides):
    """
    Train a fresh model with specified config overrides.
    Overrides: use_cad, use_eacl, use_focal, vis_backbone, vis_pool_mode
    """
    import importlib, copy
    # Apply overrides to module globals
    g = globals()
    old = {k: g[k] for k in overrides}
    g.update(overrides)

    # Re-load vis features if backbone changed
    nonlocal_vis_dim = VIS_SEM_DIM
    if 'vis_backbone' in overrides:
        if overrides['vis_backbone'] == 'clip':
            vf = torch.load(os.path.join(FEAT_DIR, 'video_clip_temporal.pt'), weights_only=False)
            nonlocal_vis_dim = 768
        else:
            vf = torch.load(os.path.join(FEAT_DIR, 'video_siglip2_temporal.pt'), weights_only=False)
            nonlocal_vis_dim = 1152
        # Rebuild dialogues with new vis features
        for d in train_dias + test_dias:
            uids = ...  # would need to re-build — skip for now; note in output
        print(f"  [note] vis_backbone change requires re-running build_dialogues — skip here")
        g.update(old)
        return

    seed_everything(42)
    m  = CIEI_MERC(nonlocal_vis_dim).to(device)
    fl = CBFocalLoss(class_counts, gamma=focal_gamma).to(device)
    el = EACLoss(N_CLASSES, embed_dim=hidden_dim).to(device)
    opt = torch.optim.AdamW(list(m.parameters()) + list(el.parameters()), lr=lr, weight_decay=weight_decay)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-6)

    best_wf1, pat = 0.0, 0
    for ep in range(1, epochs + 1):
        # Train
        m.train(); random.shuffle(train_dias)
        for dia in train_dias:
            lbs = dia['labels'].to(device)
            opt.zero_grad()
            lg, em = m.forward_dialogue(dia, return_embeds=True)
            loss = torch.tensor(0., device=device)
            if g['use_focal']: loss = loss + focal_weight * fl(lg, lbs)
            else: loss = loss + F.cross_entropy(lg, lbs)
            if g['use_eacl']: loss = loss + eacl_weight * el(em, lbs)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(list(m.parameters()) + list(el.parameters()), 1.0)
            opt.step()
        sch.step()
        # Val
        m.eval(); pred, true = [], []
        with torch.no_grad():
            for dia in test_dias:
                lg = m.forward_dialogue(dia)
                pred.extend(lg.argmax(-1).cpu().tolist())
                true.extend(dia['labels'].tolist())
        wf1 = f1_score(true, pred, average='weighted') * 100
        if wf1 > best_wf1: best_wf1, pat = wf1, 0
        else:
            pat += 1
            if pat >= patience: break

    g.update(old)
    print(f"  [{label}]  WF1={best_wf1:.2f}%")
    return best_wf1

# Run ablations
print("Running ablations (each trains from scratch — may take a while)\n")
results = {}
results['full']         = run_ablation('Full model (CAD + EACL + Focal)')
results['no_cad']       = run_ablation('No CAD (explicit edges only)',  use_cad=False)
results['no_eacl']      = run_ablation('No EACL',                       use_eacl=False)
results['no_focal']     = run_ablation('No CB-Focal (plain CE)',        use_focal=False)
results['attn_pool']    = run_ablation('vis_pool=attn',                  vis_pool_mode='attn')
results['concat_pool']  = run_ablation('vis_pool=concat',               vis_pool_mode='concat')
print("\nAblation summary:")
for k, v in results.items():
    if v is not None: print(f"  {k:<25} WF1={v:.2f}%")